In [1]:
import polars as pl
import json
import numpy as np

df = pl.read_parquet('/rxrx/data/microscopy/photosynthetic_metadata/download_day_2perc_val.parquet')

# any data without a path to an image is useless
num_to_drop = df.filter(pl.col("path").is_null()).shape[0]
print(f"Dropping {num_to_drop} rows with null paths")
df = df.filter(~pl.col("path").is_null())

# drop rows with shape != [2048, 2048, 6]
num_to_drop = df.filter(pl.col("shape") != [2048, 2048, 6]).shape[0]
print(f"Dropping {num_to_drop:,} rows with shape != [2048, 2048, 6]")
df = df.filter(pl.col("shape") == [2048, 2048, 6])

# drop non-cellpaint images
num_to_drop = df.filter(pl.col("image_type") != "cellpaint").shape[0]
print(f"Dropping {num_to_drop:,} rows with image_type != cellpaint")
df = df.filter(pl.col("image_type") == "cellpaint")

print(f"Remaining rows: {df.shape[0]:,}")

rxrx1 = pl.read_parquet('/rxrx/data/microscopy/photosynthetic_metadata/rxrx1.parquet')
drop_rxrx1_experiments = rxrx1.filter((rxrx1['cell_type'] == "HepG2") | (rxrx1['cell_type'] == "U2OS"))['experiment'].unique()
num_to_drop = df.filter(df['experiment'].is_in(drop_rxrx1_experiments)).shape[0]
print(f"Dropping {num_to_drop:,} rows with rxrx1 experiments")
df = df.filter(~df['experiment'].is_in(drop_rxrx1_experiments))

print(f"Remaining rows: {df.shape[0]:,}")

drugscreen_path = "/mnt/ps/home/CORP/charlie.jones/project/project-zeus/experiments_drugscreen_map-2023-08-28.json"
drugscreen_exps = [exp['experiment_label'] for exp in json.load(open(drugscreen_path))]
print("Dropping experiments that are drugscreens and have more than one rec_id")
tmp = df.with_columns(
    pl.col("rec_cleaned").list.len().alias("rec_cleaned_length")
).filter(((pl.col("rec_cleaned_length") > 1) & (pl.col("experiment").is_in(drugscreen_exps))))
print(f"Remaining rows: {df.shape[0]:,}")

print("Dropping 1000 random compounds from the first 20_000 compounds Recursion screened")
drop_list = [f"REC-{i:07d}" for i in np.random.RandomState(42).choice(20_000, size=1000, replace=False)]
df = df.filter(
    pl.col("rec_cleaned").list.set_intersection(drop_list).list.len() == 0
)
print(f"Remaining rows: {df.shape[0]:,}")


Dropping 0 rows with null paths
Dropping 2,505,428 rows with shape != [2048, 2048, 6]
Dropping 691,442 rows with image_type != cellpaint
Remaining rows: 12,418,461
Dropping 0 rows with rxrx1 experiments


/tmp/ipykernel_1554764/1619228375.py:26: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  num_to_drop = df.filter(df['experiment'].is_in(drop_rxrx1_experiments)).shape[0]
/tmp/ipykernel_1554764/1619228375.py:28: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  df = df.filter(~df['experiment'].is_in(drop_rxrx1_experiments))


Remaining rows: 12,418,461
Dropping experiments that are drugscreens and have more than one rec_id
Remaining rows: 12,418,461
Dropping 1000 random compounds from the first 20_000 compounds Recursion screened
Remaining rows: 12,309,318


In [2]:
# dropping reserved experiments and rec_ids from benchmarking team 

import json

with open('/rxrx/data/valence/internal_benchmarking/vcds1/v1_reservation.json') as f:
    reserved = json.load(f)

reserved_experiments = set(reserved["plated_rxrx_labels"])
reserved_rec_ids = set(reserved["rec_ids"])

num_to_drop = df.filter( (pl.col("experiment").is_in(reserved_experiments)))
print(f"Dropping {num_to_drop.shape[0]} rows with reserved experiments")
df = df.filter(~pl.col("experiment").is_in(reserved_experiments))

num_to_drop = df.filter( (pl.col("rec_cleaned").list.set_intersection(list(reserved_rec_ids)).list.len() > 0))
print(f"Dropping {num_to_drop.shape[0]} rows with reserved rec_ids")
df = df.filter(~pl.col("rec_cleaned").list.set_intersection(list(reserved_rec_ids)).list.len() > 0)

print(f"Remaining rows: {df.shape[0]:,}")


Dropping 159369 rows with reserved experiments
Dropping 391619 rows with reserved rec_ids
Remaining rows: 12,149,949


In [3]:
from sklearn.preprocessing import LabelEncoder

# add integer encoding of perturbation, experiment, cell_type to train with
df = df.with_columns(pl.Series(name="perturbation_id", values=LabelEncoder().fit(df["perturbation"].unique()).transform(df["perturbation"])))
df = df.with_columns(pl.Series(name="experiment_id", values=LabelEncoder().fit(df["experiment"].unique()).transform(df["experiment"])))
df = df.with_columns(pl.Series(name="cell_type_id", values=LabelEncoder().fit(df["cell_type"].unique()).transform(df["cell_type"])))

assert len(df["perturbation_id"].unique()) == len(df["perturbation"].unique())
assert len(df["experiment_id"].unique()) == len(df["experiment"].unique())
assert len(df["cell_type_id"].unique()) == len(df["cell_type"].unique())

## Summary stats

In [4]:
print(df["image_type"].value_counts(sort=True))
print(df["shape"].value_counts(sort=True))
print(df["perturbation_type"].value_counts(sort=True))
print(df["cell_type"].value_counts(sort=True))

shape: (1, 2)
┌────────────┬──────────┐
│ image_type ┆ count    │
│ ---        ┆ ---      │
│ str        ┆ u32      │
╞════════════╪══════════╡
│ cellpaint  ┆ 12149949 │
└────────────┴──────────┘
shape: (1, 2)
┌─────────────────┬──────────┐
│ shape           ┆ count    │
│ ---             ┆ ---      │
│ list[u64]       ┆ u32      │
╞═════════════════╪══════════╡
│ [2048, 2048, 6] ┆ 12149949 │
└─────────────────┴──────────┘
shape: (27, 2)
┌─────────────────────────────────┬─────────┐
│ perturbation_type               ┆ count   │
│ ---                             ┆ ---     │
│ list[str]                       ┆ u32     │
╞═════════════════════════════════╪═════════╡
│ ["COMPOUND", "GUIDE"]           ┆ 5394501 │
│ ["GUIDE"]                       ┆ 4924248 │
│ ["COMPOUND"]                    ┆ 1373581 │
│ []                              ┆ 196747  │
│ ["COMPOUND", "SIRNA"]           ┆ 103288  │
│ …                               ┆ …       │
│ ["SIRNA", "SIRNA"]              ┆ 90      │
│ ["CO

In [5]:
print(df["experiment"].value_counts(sort=True).describe())

shape: (9, 3)
┌────────────┬────────────────────────────┬─────────────┐
│ statistic  ┆ experiment                 ┆ count       │
│ ---        ┆ ---                        ┆ ---         │
│ str        ┆ str                        ┆ f64         │
╞════════════╪════════════════════════════╪═════════════╡
│ count      ┆ 4106                       ┆ 4106.0      │
│ null_count ┆ 0                          ┆ 0.0         │
│ mean       ┆ null                       ┆ 2959.071846 │
│ std        ┆ null                       ┆ 7605.384769 │
│ min        ┆ 1536-puddin01-H-p1-b       ┆ 1.0         │
│ 25%        ┆ null                       ┆ 14.0        │
│ 50%        ┆ null                       ┆ 226.0       │
│ 75%        ┆ null                       ┆ 1634.0      │
│ max        ┆ pheno_optimus2_ASTRO_p3_5d ┆ 102426.0    │
└────────────┴────────────────────────────┴─────────────┘


In [6]:
print(df["perturbation"].value_counts(sort=True).describe())

shape: (9, 3)
┌────────────┬─────────────────────────────────┬────────────┐
│ statistic  ┆ perturbation                    ┆ count      │
│ ---        ┆ ---                             ┆ ---        │
│ str        ┆ str                             ┆ f64        │
╞════════════╪═════════════════════════════════╪════════════╡
│ count      ┆ 624779                          ┆ 624779.0   │
│ null_count ┆ 0                               ┆ 0.0        │
│ mean       ┆ null                            ┆ 19.446795  │
│ std        ┆ null                            ┆ 293.589607 │
│ min        ┆ rec_id=;concentration=          ┆ 1.0        │
│ 25%        ┆ null                            ┆ 8.0        │
│ 50%        ┆ null                            ┆ 9.0        │
│ 75%        ┆ null                            ┆ 13.0       │
│ max        ┆ rec_id=REC-SIRNA-070391;concen… ┆ 196747.0   │
└────────────┴─────────────────────────────────┴────────────┘


## Creating splits

In [7]:
# add id column to identify each row
df = df.with_columns(pl.Series(name="id", values=range(df.shape[0])))

withheld_ids = df.sample(n=32768, seed=42, with_replacement=False, shuffle=True).select("id")

# add split col to df with values 'train', 'iid_cellpaint', 'iid_brightfield'
df = df.with_columns(
    pl.when(pl.col("id").is_in(pl.Series(withheld_ids)))
    .then(pl.lit("val_iid"))
    .otherwise(pl.lit("train"))
    .alias("split")
)

print(df["split"].value_counts(sort=True))



/tmp/ipykernel_1554764/772701627.py:7: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  df = df.with_columns(


shape: (2, 2)
┌─────────┬──────────┐
│ split   ┆ count    │
│ ---     ┆ ---      │
│ str     ┆ u32      │
╞═════════╪══════════╡
│ train   ┆ 12117181 │
│ val_iid ┆ 32768    │
└─────────┴──────────┘


## Saving


In [8]:
import os

os.makedirs("../metadata", exist_ok=True)
df.write_parquet("../metadata/modest_cell2.parquet")